Vector Search Without a Vector Database

Build the smallest possible semantic search engine yourself to understand retrieval before using a vector database.

## Goals
1. Create document chunks.
2. Embed every chunk.
3. Keep vectors in memory.
4. Embed a user query.
5. Calculate similarity between the query and every chunk.
6. Sort by similarity.
7. Return the top K chunks.

**Milestone:** Understand retrieval independently of any database or framework. No LangChain or vector database is used.

## 1. Install the required libraries

- sentence-transformers 

- numpy

## 2. Load the embedding model
We'll use `all-MiniLM-L6-v2`, which produces 384-dimensional embeddings.

In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


## 3. Create document chunks
Imagine these chunks were extracted from an employee policy document.

In [9]:
chunks = [
    "Casual leave is 12 days per year.",
    "Sick leave is 10 days per year.",
    "Employees must submit expenses through the portal.",
    "Sick leave will not be carry forward to the next year.",
]

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: {chunk}")

Chunk 0: Casual leave is 12 days per year.
Chunk 1: Sick leave is 10 days per year.
Chunk 2: Employees must submit expenses through the portal.
Chunk 3: Sick leave will not be carry forward to the next year.


## 4. Embed every chunk
Each chunk becomes a vector. The vectors stay in a NumPy array in memory.

In [10]:
chunk_embeddings = model.encode(chunks)

print("Embedding matrix shape:", chunk_embeddings.shape)
print("Rows = chunks; columns = embedding dimensions.")

Embedding matrix shape: (4, 384)
Rows = chunks; columns = embedding dimensions.


## 5. Define cosine similarity
A higher cosine-similarity score means the vectors are more similar according to the model. The score is not a probability.

In [4]:
def cosine_similarity(a, b):
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    if denominator == 0:
        return 0.0
    return float(np.dot(a, b) / denominator)

## 6. Build the semantic search function

The chunks are embedded once. For each new query, we embed only the query, compare it with stored chunk vectors, sort descending, and return the top `k`.

In [5]:
def semantic_search(query, chunks, chunk_embeddings, top_k=2):
    query_embedding = model.encode(query)

    scored_chunks = []
    for index, chunk_embedding in enumerate(chunk_embeddings):
        score = cosine_similarity(query_embedding, chunk_embedding)
        scored_chunks.append({
            "chunk": chunks[index],
            "score": score,
            "index": index
        })

    scored_chunks.sort(key=lambda item: item["score"], reverse=True)
    return scored_chunks[:max(0, top_k)]

## 7. Search for sick-leave information

In [17]:
#query = "How many sick days can an employee take?"
query = "Explain Sick leave policy."


results = semantic_search(query, chunks, chunk_embeddings, top_k=2)

print("Query:", query)
for rank, result in enumerate(results, start=1):
    print(f"{rank}. Score: {result['score']:.4f}")
    print(f"   Chunk: {result['chunk']}")

Query: Explain Sick leave policy.
1. Score: 0.7055
   Chunk: Sick leave is 10 days per year.
2. Score: 0.6497
   Chunk: Sick leave will not be carry forward to the next year.


## 8. Try another query
Change the query to search for the expense process.

In [18]:
query = "Where should employees submit their expenses?"

results = semantic_search(query, chunks, chunk_embeddings, top_k=2)
for rank, result in enumerate(results, start=1):
    print(f"{rank}. ({result['score']:.4f}) {result['chunk']}")

1. (0.7868) Employees must submit expenses through the portal.
2. (0.2200) Sick leave is 10 days per year.


## 9. Experiment with top K
`top_k=1` returns one best match. If K is larger than the number of chunks, all available chunks are returned.

In [8]:
query = "What is the casual leave allowance?"

for k in [1, 2, 3, 10]:
    results = semantic_search(query, chunks, chunk_embeddings, top_k=k)
    print(f"\ntop_k={k} -> {len(results)} result(s)")
    for result in results:
        print(f"  {result['score']:.4f}: {result['chunk']}")


top_k=1 -> 1 result(s)
  0.7222: Casual leave is 12 days per year.

top_k=2 -> 2 result(s)
  0.7222: Casual leave is 12 days per year.
  0.5293: Sick leave is 10 days per year.

top_k=3 -> 3 result(s)
  0.7222: Casual leave is 12 days per year.
  0.5293: Sick leave is 10 days per year.
  0.2662: Employees must submit expenses through the portal.

top_k=10 -> 3 result(s)
  0.7222: Casual leave is 12 days per year.
  0.5293: Sick leave is 10 days per year.
  0.2662: Employees must submit expenses through the portal.


## 10. Retrieval flow

```text
Policy chunks
    ↓
Embedding model → chunk vectors stored in Python memory
                              ↑
User query → embedding model → query vector
                              ↓
                  similarity with each chunk
                              ↓
                    sort scores descending
                              ↓
                        return top K
```

## Retrieval vs. complete RAG

This notebook retrieves relevant text. A complete RAG pipeline adds a language model after retrieval: pass the retrieved chunks as context and ask the model to answer grounded in that context.

A vector database is not needed for this small example. For larger collections, a vector index/database can improve search efficiency and provide persistence or metadata filtering.